# MTLnet — Colab Training

Multi-task learning for POI prediction (category + next-POI).

**Typical workflow:**
1. Run **Setup** (sections 1–5) once per session
2. Set your **Pipeline Configuration** (section 6)
3. Run **Step 1** → Generate Embeddings
4. Run **Step 2** → Create Model Inputs
5. Run **Step 3** → Train

**Drive layout expected:**
```
MyDrive/mestrado/PoiMtlNet/
  data/checkins/      ← raw checkins (Florida.parquet, Alabama.parquet, …)
  data/miscellaneous/ ← shapefiles (tl_2022_*_tract_*/)
  output/             ← generated embeddings and model inputs  [auto-created]
  results/            ← training results                        [auto-created]
```

---
## ① Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
from pathlib import Path

DRIVE_ROOT   = Path('/content/drive/MyDrive/mestrado/PoiMtlNet')
DATA_ROOT    = DRIVE_ROOT / 'data'
OUTPUT_DIR   = DRIVE_ROOT / 'output'
RESULTS_ROOT = DRIVE_ROOT / 'results'

for p in [DATA_ROOT, OUTPUT_DIR, RESULTS_ROOT]:
    p.mkdir(parents=True, exist_ok=True)

os.environ['DATA_ROOT']    = str(DATA_ROOT)
os.environ['OUTPUT_DIR']   = str(OUTPUT_DIR)
os.environ['RESULTS_ROOT'] = str(RESULTS_ROOT)

print(f'DATA_ROOT:    {DATA_ROOT}  (exists={DATA_ROOT.exists()})')
print(f'OUTPUT_DIR:   {OUTPUT_DIR}  (exists={OUTPUT_DIR.exists()})')
print(f'RESULTS_ROOT: {RESULTS_ROOT}  (exists={RESULTS_ROOT.exists()})')

In [ ]:
REPO_DIR = Path('/content/PoiMtlNet')

if not REPO_DIR.exists():
    !git clone https://github.com/VitorHugoOli/PoiMtlNet.git {REPO_DIR}
else:
    !git -C {REPO_DIR} pull

%cd {REPO_DIR}
!git log --oneline -3

In [ ]:
# Base requirements
!pip install -q -r requirements_colab.txt

# NashMTL requires cvxpy + ECOS solver
!pip install -q cvxpy

# PyTorch Geometric — needed for HGI, DGI, POI2HGI, Check2HGI
!pip install -q torch-geometric

In [ ]:
import sys

for sub in ('src', 'research'):
    p = str(REPO_DIR / sub)
    if p not in sys.path:
        sys.path.insert(0, p)

from configs.globals import DEVICE
print('Device:', DEVICE)

---
## ② Pipeline Configuration

**Edit this cell before running any pipeline step.**  
All subsequent cells read `STATE`, `ENGINE`, and `TASK` from here.

In [ ]:
# ── edit here ─────────────────────────────────────────────────────────────────
STATE  = 'florida'   # florida | alabama | georgia | california | texas | arizona
ENGINE = 'hgi'       # hgi | dgi | poi2hgi | check2hgi | time2vec | space2vec | sphere2vec | fusion
TASK   = 'mtl'       # mtl | category | next

# Optional training overrides (None = use config defaults)
EPOCHS = None        # e.g. 50
FOLDS  = None        # e.g. 1 for a quick smoke test
# ──────────────────────────────────────────────────────────────────────────────

from configs.paths import EmbeddingEngine, IoPaths, Resources
ENGINE_ENUM = EmbeddingEngine(ENGINE)

# Map state name to shapefile resource (needed by graph-based engines)
_SHAPEFILES = {
    'florida':    Resources.TL_FL,
    'alabama':    Resources.TL_AL,
    'georgia':    Resources.TL_GA,
    'california': Resources.TL_CA,
    'texas':      Resources.TL_TX,
    'arizona':    Resources.TL_AZ,
}
SHAPEFILE = _SHAPEFILES.get(STATE.lower())

print(f'State:     {STATE}')
print(f'Engine:    {ENGINE}')
print(f'Task:      {TASK}')
print(f'Shapefile: {SHAPEFILE}')
print(f'Epochs:    {EPOCHS or "(default)"}')
print(f'Folds:     {FOLDS  or "(all)"}')

---
## ③ Step 1 — Generate Embeddings

Run the subsection that matches your `ENGINE`.  
Skip this step entirely if embeddings are already in `output/` on Drive.

### HGI / POI2HGI / Check2HGI  (`engine = hgi | poi2hgi | check2hgi`)

In [ ]:
# Run only if ENGINE in ('hgi', 'poi2hgi', 'check2hgi')
assert ENGINE in ('hgi', 'poi2hgi', 'check2hgi'), f'Wrong engine: {ENGINE}'
assert SHAPEFILE is not None, f'No shapefile registered for state "{STATE}"'

import importlib.util
from copy import copy

# Load the correct pipeline module without executing __main__
_pipe_path = REPO_DIR / f'pipelines/embedding/{ENGINE}.pipe.py'
_spec = importlib.util.spec_from_file_location('_pipe', _pipe_path)
_pipe = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(_pipe)

# Patch STATES to run only our chosen state
_pipe.STATES = {
    STATE.capitalize(): {'shapefile': SHAPEFILE},
}

_pipe.run_pipeline()

### DGI  (`engine = dgi`)

In [ ]:
assert ENGINE == 'dgi', f'Wrong engine: {ENGINE}'
assert SHAPEFILE is not None, f'No shapefile registered for state "{STATE}"'

import importlib.util

_pipe_path = REPO_DIR / 'pipelines/embedding/dgi.pipe.py'
_spec = importlib.util.spec_from_file_location('_pipe', _pipe_path)
_pipe = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(_pipe)

_pipe.STATES = {
    STATE.capitalize(): {'shapefile': SHAPEFILE},
}

_pipe.run_pipeline()

### Time2Vec  (`engine = time2vec`)

In [ ]:
assert ENGINE == 'time2vec', f'Wrong engine: {ENGINE}'

import importlib.util

_pipe_path = REPO_DIR / 'pipelines/embedding/time2vec.pipe.py'
_spec = importlib.util.spec_from_file_location('_pipe', _pipe_path)
_pipe = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(_pipe)

_pipe.STATES = {STATE.capitalize(): {}}

_pipe.run_pipeline()

### Space2Vec / Sphere2Vec  (`engine = space2vec | sphere2vec`)

In [ ]:
assert ENGINE in ('space2vec', 'sphere2vec'), f'Wrong engine: {ENGINE}'

import importlib.util

_pipe_path = REPO_DIR / f'pipelines/embedding/{ENGINE}.pipe.py'
_spec = importlib.util.spec_from_file_location('_pipe', _pipe_path)
_pipe = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(_pipe)

_pipe.STATES = {STATE.capitalize(): {}}

_pipe.run_pipeline()

### Fusion  (`engine = fusion`)

Fusion concatenates multiple pre-existing embeddings — run after each component engine is done.

In [ ]:
assert ENGINE == 'fusion', f'Wrong engine: {ENGINE}'

import importlib.util

_pipe_path = REPO_DIR / 'pipelines/fusion.pipe.py'
_spec = importlib.util.spec_from_file_location('_pipe', _pipe_path)
_pipe = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(_pipe)

_pipe.STATES = {STATE.capitalize(): {}}

_pipe.run_pipeline()

---
## ④ Step 2 — Create Model Inputs

Generates `category.parquet` and `next.parquet` inside `output/{engine}/{state}/input/`.
Skip if inputs are already present.

In [ ]:
import importlib.util

_pipe_path = REPO_DIR / 'pipelines/create_inputs.pipe.py'
_spec = importlib.util.spec_from_file_location('_pipe', _pipe_path)
_pipe = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(_pipe)

_pipe.STATES = {
    STATE.capitalize(): {
        'engine': ENGINE_ENUM,
        'use_checkin': ENGINE == 'time2vec',  # time2vec uses check-in-level embeddings
    }
}

_pipe.run_pipeline()

In [ ]:
# Verify inputs exist before training
cat_path  = IoPaths.get_category(STATE, ENGINE_ENUM)
next_path = IoPaths.get_next(STATE, ENGINE_ENUM)

print(f'Category input  ({"OK" if cat_path.exists() else "MISSING"}): {cat_path}')
print(f'Next-POI input  ({"OK" if next_path.exists() else "MISSING"}): {next_path}')

---
## ⑤ Step 3 — Train

Results (metrics CSVs, classification reports, plots, checkpoints) are written to Drive under `results/`.

In [ ]:
# Build the training command from the config set in section ②
_cmd = f'python scripts/train.py --task {TASK} --state {STATE} --engine {ENGINE}'
if EPOCHS is not None:
    _cmd += f' --epochs {EPOCHS}'
if FOLDS is not None:
    _cmd += f' --folds {FOLDS}'

print('Command:', _cmd)

In [ ]:
# Full run
!{_cmd}

In [ ]:
# Smoke test — 1 fold, 5 epochs (confirm setup is correct before the full run)
!python scripts/train.py --task {TASK} --state {STATE} --engine {ENGINE} --folds 1 --epochs 5

---
## ⑥ Inspect Results

In [ ]:
import pandas as pd

results_dir = IoPaths.get_results_dir(STATE, ENGINE_ENUM)
print('Results directory:', results_dir)
print()

csv_files = sorted(results_dir.glob('*.csv'))
if csv_files:
    for f in csv_files:
        print(f'--- {f.name} ---')
        print(pd.read_csv(f).to_string(index=False))
        print()
else:
    print('No CSV results found yet.')